In [6]:
#Tarea 9.5 Martín Alvarez

import numpy as np

class MinStep(Exception):
    pass

def rkf(f, a, b, yo, tol, hmin, hmax):
    
    t = a
    w = yo
    h = hmax
    flag = 1
    
    ts = [t]
    sol = [w]
    hs = []
    rs = []
    
    while flag:
        
        # Evaluaciones de la función 
        k1 = h*f(t, w)
        k2 = h*f(t + h/4, w + k1/4)
        k3 = h*f(t + 3*h/8, w + 3*k1/32 + 9*k2/32)
        k4 = h*f(t + 12*h/13, w + 1932*k1/2197 - 7200*k2/2197 + 7296*k3/2197)
        k5 = h*f(t + h, w + 439*k1/216 - 8*k2 + 3680*k3/513 - 845*k4/4104)
        k6 = h*f(t + h/2, w - 8*k1/27 + 2*k2 - 3544*k3/2565 + 1859*k4/4104 - 11*k5/40)
        
        # Estima del error dada por la diferencia entre dos RKs
        R = abs(k1/360 - 128*k3/4275 - 2197*k4/75240 + k5/50 + 2*k6/55)
        
        # Si el error es menor que la tolerancia, se acepta el paso
        if R <= tol:
            t += h
            w = w + 25*k1/216 + 1408*k3/2565 + 2197*k4/4104 - k5/5
            ts.append(t)
            sol.append(w)
            hs.append(h)
            rs.append(R)
            
        # el 0.84 es un factor para hacer la estima más conservadora 
        # corresponde a 1/2**(1/4)
        q = 0.84*(tol/R)**(1/4)
        
        # ACTUALIZACIÓN DE h
        # Se establece un q mínimo para no quedarse atrapado en alguna región.
        # Esto se puede cambiar.
        if q <= 0.1:
            h = 0.1*h
        # Se establece también un q máximo para no peder detalles de la función
        elif q >= 4:
            h = 4*h
        else:
            h = q*h
        if h > hmax:
            h = hmax
        
        # Si nos pasamos de b, terminamos
        if t >= b:
            flag = 0
        # El último paso debe llegar a b
        elif t + h > b:
            h = b - t
        elif h < hmin:
            raise MinStep('Se excedió el paso mínimo.')
            
    return np.array(ts), np.array(sol), np.array(hs), np.array(rs)


def f(t, y):
    return 10*y - 10*t**(-2) - 2*t**(-3)

In [5]:
for yo in [2, 1, 0.1]:
    exacto = 0.25 + (yo - 1)*np.exp(10)
    print("y(1) =", yo, " exacto y(2) =", exacto)
    for tol in [0.01, 0.001, 0.0001, 10**(-5)]:
        ts, sol, hs, rs = rkf(f, 1, 2, yo, tol, 1e-8, 0.25)
        error = abs(sol[-1] - exacto)
        print(tol, len(ts), sol[-1], error)

y(1) = 2  exacto y(2) = 22026.715794806718
0.01 22 21308.963203102692 717.7525917040257
0.001 37 22028.47291043788 1.7571156311605591
0.0001 59 22028.000067143177 1.284272336459253
1e-05 95 22027.040611380096 0.32481657337848446
y(1) = 1  exacto y(2) = 0.25
0.01 6 -18.404407611726267 18.654407611726267
0.001 7 -3.964252543939562 4.214252543939562
0.0001 9 0.18742028837646177 0.06257971162353823
1e-05 14 0.2941586968953065 0.04415869689530649
y(1) = 0.1  exacto y(2) = -19823.569215326046
0.01 21 -19215.001939124566 608.5672762014801
0.001 36 -19824.55433236275 0.9851170367037412
0.0001 58 -19824.75149145535 1.1822761293042277
1e-05 93 -19823.853939960693 0.28472463464640896
